# ⚽ FIFA World Cup 2026 — Live Dashboard

A **Spur App** that fuses two live data sources into one reactive dashboard:

- **Polymarket** (`gamma-api.polymarket.com`) — prediction-market *implied odds* for World Cup 2026 questions (winner, hosts, golden boot).
- **RSSHub** (`rsshub.app`, Google-News fallback) — the World Cup headline feed.

**Pipeline (reactive DAG):** two Python *source* cells pull each feed and publish an Arrow **port** (`wc_markets`, `wc_news`); the **Deno frontend cell** reads those ports and renders a **Perspective** datagrid + chart, with the live MCP tool `wc_snapshot` preferred when the plugin is running.

> Re-run a source cell (or arm a cron schedule on it) → the cascade re-renders the dashboard. In **App mode**, only the frontend cell is shown. Implied probabilities are *market prices*, not forecasts.

In [12]:
import os, sys

# Reuse the app's own data layer (server/worldcup.py) — the exact code the
# wc_markets / wc_snapshot MCP tools call. Single source of truth.
for _cand in (os.path.join(os.getcwd(), "server"),
              os.path.join(os.path.dirname(os.getcwd()), "server")):
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand)
        break
import worldcup

# --- Polymarket data source -------------------------------------------------
markets = worldcup.fetch_markets("world cup", 50)
live = bool(markets)
if not markets:
    # Offline / no active market yet — labelled sample so the dashboard renders.
    _sample = [
        ("[sample] Will Spain win the 2026 World Cup?",     "0.18", "4200000"),
        ("[sample] Will Argentina win the 2026 World Cup?", "0.15", "3900000"),
        ("[sample] Will France win the 2026 World Cup?",    "0.14", "3500000"),
        ("[sample] Will Brazil win the 2026 World Cup?",    "0.13", "3300000"),
        ("[sample] Will England win the 2026 World Cup?",   "0.11", "2600000"),
    ]
    markets = [
        worldcup.shape_market(
            {"question": q, "outcomes": '["Yes","No"]',
             "outcomePrices": f'["{p}","{round(1 - float(p), 2)}"]',
             "volume": v, "slug": ""}
        )
        for q, p, v in _sample
    ]

# Prove the Polymarket spur_rest DuckDB datasource is wired (cheap LIMIT 1 probe
# — never count(*), which would force the table function to paginate the whole
# active-market set).
try:
    import duckdb
    _ext = os.path.expanduser("~/.spur/extensions/spur_rest.duckdb_extension")
    _con = duckdb.connect(config={"allow_unsigned_extensions": "true"})
    _con.execute(f"LOAD '{_ext}'")
    _con.execute("SELECT question FROM polymarket_markets() LIMIT 1").fetchall()
    print("polymarket datasource OK: spur_rest table function reachable")
    _con.close()
except Exception as _e:  # noqa: BLE001
    print(f"(polymarket datasource probe skipped: {_e})")

spur.put("wc_markets", markets)
print(f"{len(markets)} World Cup markets ({'LIVE' if live else 'SAMPLE'}) -> port wc_markets")
markets[:3]

polymarket datasource OK: spur_rest table function reachable


question,outcome,yes_prob,implied_pct,volume,volume_24hr,liquidity,end_date,url
Will Scotland advance to the knockout stages at the 2026 FIFA World Cup?,Yes,0.745,74.5,99660.16080399985,29396.35715700001,6019.1341,2026-06-28,https://polymarket.com/market/will-scotland-advance-to-the-knockout-stages-at-the-2026-fifa-world-cup
Will France reach the Round of 16 at the 2026 FIFA World Cup?,Yes,0.835,83.5,99588.6458620001,10939.258818,27115.4829,2026-07-04,https://polymarket.com/market/will-france-reach-the-round-of-16-at-the-2026-fifa-world-cup-20260602025120767
Will Haiti advance to the knockout stages at the 2026 FIFA World Cup?,Yes,0.0465,4.7,99542.24055700006,11558.092895,51623.91432,2026-06-28,https://polymarket.com/market/will-haiti-advance-to-the-knockout-stages-at-the-2026-fifa-world-cup
Will Spain reach the Quarterfinals at the 2026 FIFA World Cup?,Yes,0.625,62.5,99005.08936999996,11463.205815,53014.9881,,https://polymarket.com/market/will-spain-reach-the-quarterfinals-at-the-2026-fifa-world-cup-20260602145134495
Will Jamal Musiala record the most goal contributions at the 2026 FIFA World Cup?,Yes,0.161,16.1,999.7510820000001,93.290273,5727.04625,2026-07-20,https://polymarket.com/market/will-jamal-musiala-record-the-most-goal-contributions-at-the-2026-fifa-world-cup


13 World Cup markets (LIVE) -> port wc_markets


[{'question': 'Will Scotland advance to the knockout stages at the 2026 FIFA World Cup?',
  'outcome': 'Yes',
  'yes_prob': 0.745,
  'implied_pct': 74.5,
  'volume': 99660.16080399985,
  'volume_24hr': 29396.35715700001,
  'liquidity': 6019.1341,
  'end_date': '2026-06-28',
  'url': 'https://polymarket.com/market/will-scotland-advance-to-the-knockout-stages-at-the-2026-fifa-world-cup'},
 {'question': 'Will France reach the Round of 16 at the 2026 FIFA World Cup?',
  'outcome': 'Yes',
  'yes_prob': 0.835,
  'implied_pct': 83.5,
  'volume': 99588.6458620001,
  'volume_24hr': 10939.258818,
  'liquidity': 27115.4829,
  'end_date': '2026-07-04',
  'url': 'https://polymarket.com/market/will-france-reach-the-round-of-16-at-the-2026-fifa-world-cup-20260602025120767'},
 {'question': 'Will Haiti advance to the knockout stages at the 2026 FIFA World Cup?',
  'outcome': 'Yes',
  'yes_prob': 0.0465,
  'implied_pct': 4.7,
  'volume': 99542.24055700006,
  'volume_24hr': 11558.092895,
  'liquidity': 5

In [11]:
import os, sys

for _cand in (os.path.join(os.getcwd(), "server"),
              os.path.join(os.path.dirname(os.getcwd()), "server")):
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand)
        break
import worldcup

# --- RSSHub data source (rsshub.app first, Google-News RSS fallback) --------
news = worldcup.fetch_news(30)
live_news = bool(news)
if not news:
    news = [
        {"title": "[sample] World Cup 2026 host cities finalize match schedule",
         "link": "https://example.org/wc/1", "published": "", "source": "sample"},
        {"title": "[sample] Qualification race tightens across confederations",
         "link": "https://example.org/wc/2", "published": "", "source": "sample"},
        {"title": "[sample] Ticket demand sets a record for the opening match",
         "link": "https://example.org/wc/3", "published": "", "source": "sample"},
    ]

spur.put("wc_news", news)
_src = news[0]["source"] if news else "none"
print(f"{len(news)} headlines ({'LIVE via ' + _src if live_news else 'SAMPLE'}) -> port wc_news")
news[:3]

title,link,published,source
"World Cup 2026 scores, results: Japan, Netherlands thrill; Ivory Coast stuns Ecuador; Germany flexes on wild Day 4 - Yahoo Sports",https://news.google.com/rss/articles/CBMipgFBVV95cUxOUUhRSXRyM3EyNkVwck1HdlAxQkFXMzgybGo0RmY1NE5aeUtTODN5aEhKTG1faVpISVJwdnhJaTZ3ZS1hRlo1SEhhdGcycXFyY3JKaGpCV1JsZEQ2OXNrMHV3Nlc1akJMYW9qejlkZVhTWEdKaUJZQjRuWS00RzJxOWR1MDZYOVZsSEpCZEZQTC1abmFxekxhTWFSd09EZ3AzOGNyQjlR?oc=5,"Mon, 15 Jun 2026 05:28:00 GMT",rss
Trump’s World Cup czar calls early entry for Iran team a ‘goodwill gesture’ - Politico,https://news.google.com/rss/articles/CBMisgFBVV95cUxONGE3V0VfSnlVaC1nVl9PMm95VUtqSnBPc0FFU19uSnJJZWFkTmFWSUVaR1ZGaUtzMTc0MkhRYUo0Rzh5bzNHeXJSMWkyZUg2NU51SkU1MDJlSHN2Sk9Tb3BnWHgzOHN1djFkTmxBOFE3Nnh1dm83blZMZmJBWXpUaEQ5R0h5RHVEWmU3aFNZNkh1Qml6TVlDUEprZVAwaEtCaFk5MkpOYm5RSDUyd1RXOGp3?oc=5,"Sun, 14 Jun 2026 20:30:00 GMT",rss
EVERY ANGLE of Yasin Ayari's second GOAL 🔥 Sweden vs Tunisia 2026 FIFA World Cup™ - FOX Sports,https://news.google.com/rss/articles/CBMiYkFVX3lxTE4taXBzR25iTEs1Y2VfakNJTFVCWHY1b09JYVg1MlpiREg3My13WG1YLUVPWHhwX1k0NkFkaHg0MlIxUld3UXVzdENDVmJIbmphcUxmcWFRSWF5aDh0aHJDUGp3?oc=5,"Mon, 15 Jun 2026 05:41:41 GMT",rss
World Cup 2026: Iraola's fascinating day of World Cup scouting - BBC,https://news.google.com/rss/articles/CBMiZ0FVX3lxTE83WEktazBYQnFqTk90eU5maW1HaTlQTUlaWG5YWkVXWlJPeFlPZFFMaTZHaTlhdDJrLUIxV2xPUXcxaUVBRGMtTm9xUVNJekF3TmFLVkhTM1RrOUNBWnY0S3lLdmprZ0E?oc=5,"Mon, 15 Jun 2026 04:48:30 GMT",rss
Netherlands vs Japan Highlights | 2026 FIFA World Cup™ - FOX Sports,https://news.google.com/rss/articles/CBMiYkFVX3lxTE1vVmt6cEZET1k4aDJSZFMzTG1ELXMyR3I5TU91VEpOMkgyNkdPUTc4T1ROVC1ISXZ6U2dFOHJkUVd3STV3cjZGRFZDRVlSak43Rm9IWTlFN1d6R2RyMk9XaDJ3?oc=5,"Sun, 14 Jun 2026 22:01:17 GMT",rss


30 headlines (LIVE via rss) -> port wc_news


[{'title': 'World Cup 2026 scores, results: Japan, Netherlands thrill; Ivory Coast stuns Ecuador; Germany flexes on wild Day 4 - Yahoo Sports',
  'link': 'https://news.google.com/rss/articles/CBMipgFBVV95cUxOUUhRSXRyM3EyNkVwck1HdlAxQkFXMzgybGo0RmY1NE5aeUtTODN5aEhKTG1faVpISVJwdnhJaTZ3ZS1hRlo1SEhhdGcycXFyY3JKaGpCV1JsZEQ2OXNrMHV3Nlc1akJMYW9qejlkZVhTWEdKaUJZQjRuWS00RzJxOWR1MDZYOVZsSEpCZEZQTC1abmFxekxhTWFSd09EZ3AzOGNyQjlR?oc=5',
  'published': 'Mon, 15 Jun 2026 05:28:00 GMT',
  'source': 'rss'},
 {'title': 'Trump’s World Cup czar calls early entry for Iran team a ‘goodwill gesture’ - Politico',
  'link': 'https://news.google.com/rss/articles/CBMisgFBVV95cUxONGE3V0VfSnlVaC1nVl9PMm95VUtqSnBPc0FFU19uSnJJZWFkTmFWSUVaR1ZGaUtzMTc0MkhRYUo0Rzh5bzNHeXJSMWkyZUg2NU51SkU1MDJlSHN2Sk9Tb3BnWHgzOHN1djFkTmxBOFE3Nnh1dm83blZMZmJBWXpUaEQ5R0h5RHVEWmU3aFNZNkh1Qml6TVlDUEprZVAwaEtCaFk5MkpOYm5RSDUyd1RXOGp3?oc=5',
  'published': 'Sun, 14 Jun 2026 20:30:00 GMT',
  'source': 'rss'},
 {'title': "EVERY ANGLE of Yasin Ay

In [12]:
// World Cup 2026 — Perspective dashboard (frontend cell).
// Renders purely from the wc_markets / wc_news Arrow ports, which the two
// source cells populate reactively. To refresh, re-run a source cell (or arm a
// cron schedule on it) — the cascade re-renders this cell. We deliberately do
// NOT call an MCP tool here: that would put a blocking network + app-plugin
// round-trip in the App-mode render path (slow on open, can stall if the
// plugin is still spawning). The ports already hold the data.

function rowsOf(t) { try { return t && t.toArray ? t.toArray() : []; } catch (_) { return []; } }
function coerceMarket(r) {
  return {
    question: String(r.question ?? ""),
    implied_pct: r.implied_pct == null ? null : Number(r.implied_pct),
    volume: Number(r.volume ?? 0),
    end_date: String(r.end_date ?? ""),
    url: String(r.url ?? ""),
  };
}
function coerceNews(r) {
  return {
    title: String(r.title ?? ""),
    link: String(r.link ?? ""),
    published: String(r.published ?? ""),
    source: String(r.source ?? ""),
  };
}

const markets = rowsOf(spur.get("wc_markets")).map(coerceMarket);
const news = rowsOf(spur.get("wc_news")).map(coerceNews);

const priced = markets.filter((m) => m.implied_pct != null);
const favorite = priced.slice().sort((a, b) => b.implied_pct - a.implied_pct)[0] || null;
const totalVol = markets.reduce((s, m) => s + (m.volume || 0), 0);
const isSample = markets.some((m) => m.question.startsWith("[sample]")) || news.some((n) => n.source === "sample");

const esc = (s) => String(s == null ? "" : s).replace(/[&<>"]/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));
const fmtVol = (v) => "$" + (Number(v) || 0).toLocaleString("en-US", { maximumFractionDigits: 0 });
const fmtPct = (v) => (v == null ? "—" : Number(v).toFixed(1) + "%");

const kpiCards = [
  ["Market favorite", favorite ? esc(favorite.question.replace(/^\[sample\]\s*/, "")) : "—", favorite ? fmtPct(favorite.implied_pct) : ""],
  ["Markets tracked", String(markets.length), ""],
  ["Total volume", fmtVol(totalVol), ""],
  ["Headlines", String(news.length), ""],
].map(([label, big, sub]) => `<div class="kpi"><div class="kpi-label">${label}</div><div class="kpi-big">${big}</div><div class="kpi-sub">${sub}</div></div>`).join("");

const newsItems = news.slice(0, 18).map((n) => `<li><a href="${esc(n.link)}" target="_blank" rel="noopener">${esc(n.title)}</a><span class="src">${esc(n.source)}</span></li>`).join("") || '<li class="muted">No headlines.</li>';

// Browser-side module script. NOTE: no backticks / no ${} inside INNER so the
// outer template literal does not interpolate it.
const INNER = `
const fmtPct = (v) => (v == null ? "—" : Number(v).toFixed(1) + "%");
const fmtVol = (v) => "$" + (Number(v) || 0).toLocaleString("en-US", { maximumFractionDigits: 0 });
const esc = (s) => String(s == null ? "" : s).replace(/[&<>]/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;" }[c]));
function fallbackTable(rows) {
  let h = "<table class='tbl'><thead><tr><th>Market</th><th class='num'>Implied</th><th class='num'>Volume</th><th>Ends</th></tr></thead><tbody>";
  for (const r of rows) { h += "<tr><td>" + esc(r.question) + "</td><td class='num'>" + fmtPct(r.implied_pct) + "</td><td class='num'>" + fmtVol(r.volume) + "</td><td>" + esc(r.end_date || "") + "</td></tr>"; }
  return h + "</tbody></table>";
}
const container = document.getElementById("pv");
const data = MARKETS.length ? MARKETS : [{ question: "(no markets)", implied_pct: null, volume: 0, end_date: "", url: "" }];
let ok = false;
for (const v of ["3.7.0", "3.3.0"]) {
  try {
    const base = "https://cdn.jsdelivr.net/npm/@finos/";
    const perspective = (await import(base + "perspective@" + v + "/dist/cdn/perspective.js")).default;
    await import(base + "perspective-viewer@" + v + "/dist/cdn/perspective-viewer.js");
    await import(base + "perspective-viewer-datagrid@" + v + "/dist/cdn/perspective-viewer-datagrid.js");
    await import(base + "perspective-viewer-d3fc@" + v + "/dist/cdn/perspective-viewer-d3fc.js");
    const worker = await perspective.worker();
    const table = await worker.table(data);
    const viewer = document.createElement("perspective-viewer");
    container.innerHTML = "";
    container.appendChild(viewer);
    await viewer.load(table);
    await viewer.restore({ plugin: "Datagrid", columns: ["question", "implied_pct", "volume", "end_date"], sort: [["volume", "desc"]], theme: "Pro Dark" });
    ok = true;
    break;
  } catch (e) { /* try next version */ }
}
if (!ok) { container.classList.add("fallback"); container.innerHTML = fallbackTable(data); }
`;

const safeMarkets = JSON.stringify(markets).replace(/</g, "\\u003c");
const scriptTag = '<scr' + 'ipt type="module">\nconst MARKETS = ' + safeMarkets + ';\n' + INNER + '\n</scr' + 'ipt>';

const html = `<!doctype html><html><head><meta charset="utf-8">
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@finos/perspective-viewer@3.7.0/dist/css/themes.css">
<style>
  :root { color-scheme: dark; }
  * { box-sizing: border-box; }
  body { margin: 0; background: #0a0e1a; color: #e6edf3; font-family: Inter, system-ui, sans-serif; }
  .wrap { max-width: 1180px; margin: 0 auto; padding: 22px; }
  .top { display: flex; align-items: baseline; gap: 12px; flex-wrap: wrap; }
  h1 { font-size: 22px; margin: 0; }
  .badge { font-size: 11px; font-weight: 700; padding: 3px 9px; border-radius: 999px; letter-spacing: .04em; }
  .badge.live { background: #0f3d2e; color: #34d399; }
  .badge.sample { background: #3d2f0f; color: #fbbf24; }
  .sub { color: #93a1b3; font-size: 13px; margin: 4px 0 18px; }
  .kpis { display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; margin-bottom: 18px; }
  .kpi { background: #121829; border: 1px solid #1e2740; border-radius: 14px; padding: 14px 16px; }
  .kpi-label { color: #93a1b3; font-size: 12px; text-transform: uppercase; letter-spacing: .05em; }
  .kpi-big { font-size: 19px; font-weight: 700; margin-top: 6px; line-height: 1.25; }
  .kpi-sub { color: #34d399; font-weight: 700; margin-top: 2px; }
  .grid { display: grid; grid-template-columns: 1.7fr 1fr; gap: 16px; align-items: start; }
  .card { background: #121829; border: 1px solid #1e2740; border-radius: 14px; overflow: hidden; }
  .card h2 { font-size: 13px; text-transform: uppercase; letter-spacing: .06em; color: #93a1b3; margin: 0; padding: 14px 16px; border-bottom: 1px solid #1e2740; }
  #pv { height: 460px; width: 100%; }
  #pv.fallback { height: auto; max-height: 460px; overflow: auto; }
  .tbl { width: 100%; border-collapse: collapse; font-size: 13px; }
  .tbl th, .tbl td { text-align: left; padding: 8px 12px; border-bottom: 1px solid #1a2236; }
  .tbl th { color: #93a1b3; font-weight: 600; }
  .tbl td.num, .tbl th.num { text-align: right; font-variant-numeric: tabular-nums; }
  ul.news { list-style: none; margin: 0; padding: 4px 0; max-height: 460px; overflow: auto; }
  ul.news li { padding: 9px 16px; border-bottom: 1px solid #1a2236; font-size: 13px; display: flex; justify-content: space-between; gap: 10px; }
  ul.news a { color: #cdd9e5; text-decoration: none; }
  ul.news a:hover { color: #58a6ff; text-decoration: underline; }
  ul.news .src { color: #5b6b82; font-size: 11px; text-transform: uppercase; flex: none; }
  .muted { color: #5b6b82; }
  footer { color: #5b6b82; font-size: 11px; margin-top: 16px; }
</style></head>
<body><div class="wrap">
  <div class="top"><h1>⚽ FIFA World Cup 2026 — Live Dashboard</h1><span class="badge ${isSample ? "sample" : "live"}">${isSample ? "SAMPLE DATA" : "LIVE"}</span></div>
  <div class="sub">Polymarket implied odds × RSSHub headlines · prediction-market prices, not forecasts</div>
  <div class="kpis">${kpiCards}</div>
  <div class="grid">
    <div class="card"><h2>Markets — implied probability &amp; volume (Perspective)</h2><div id="pv">Loading Perspective…</div></div>
    <div class="card"><h2>Latest headlines</h2><ul class="news">${newsItems}</ul></div>
  </div>
  <footer>Sources: gamma-api.polymarket.com · rsshub.app (Google-News fallback). Re-run a source cell to refresh. Rendered by the world-cup-2026 Spur App.</footer>
</div>${scriptTag}</body></html>`;

await Deno.jupyter.display({ "text/html": html }, { raw: true });

<!doctype html> 
 
 
 
 ⚽ FIFA World Cup 2026 — Live Dashboard LIVE 
 Polymarket implied odds × RSSHub headlines · prediction-market prices, not forecasts 
 Market favorite Will France reach the Round of 16 at the 2026 FIFA World Cup? 83.5% Markets tracked 13 Total volume $405,844 Headlines 30 
 
 Markets — implied probability & volume (Perspective) Loading Perspective… 
 Latest headlines World Cup 2026 scores, results: Japan, Netherlands thrill; Ivory Coast stuns Ecuador; Germany flexes on wild Day 4 - Yahoo Sports rss Trump’s World Cup czar calls early entry for Iran team a ‘goodwill gesture’ - Politico rss EVERY ANGLE of Yasin Ayari's second GOAL 🔥 Sweden vs Tunisia 2026 FIFA World Cup™ - FOX Sports rss World Cup 2026: Iraola's fascinating day of World Cup scouting - BBC rss Netherlands vs Japan Highlights | 2026 FIFA World Cup™ - FOX Sports rss Japan’s Daichi Kamada scores late equalizer off corner against Netherlands | 2026 FIFA World Cup - FOX Sports rss Wheelchair-Bound Japan Fan Collects Trash After FIFA World Cup Match, Wins Internet - NDTV Sports rss World Cup 2026: guide to all 1,248 players - The Guardian rss A team-by-team guide to the 2026 World Cup: What to expect and who to watch - The Athletic - The New York Times rss World Cup 2026: a visual guide to the stadiums across the trio of host nations - The Guardian rss China didn’t qualify for the World Cup. But its fans still have a star: a card-wielding referee - CNN rss How USMNT can win World Cup group with game to spare – and why tiebreak rules raise stakes vs. Australia - The New York Times rss World Cup Daily: Sweden top Group F after Netherlands-Japan draw - ESPN rss Netherlands 2-2 Japan - FIFA rss Fifa will not punish Fox for breaking advertising rules during World Cup opener - The Guardian rss <a href="https://news.google.com/rss/articles/CBMisgFBVV95cUxQQzBhODdLUTdscXlNT3JXdG5VNTVlVS0yVDBZeGxWdV80N1BiRk4yb3M0VWV2SnJfeWotYzVmZXRrUWRyOHNUUEpuamo2NDBZSU05T0pNMElzWURwLTRManVUYjlMS3BqR2MxYl83dXh0QXVKRWlhRDdZZ2xfaElQYWtweUdzTk5vWnlzTS1UdjVySnBwbFdGT21zR1k2V3pXbmx6eGotdl9YMlZSQlYtTGNB0gG3AUFVX3lxTE1WeWtfc0xqVnYycnBBd0JOS28tZHFSbXZMSXg2Tm9oXzNQMFJfTE9Xb0hoQkNGN0czWHNkNVpWYjZrR01ZcTUxT01RT0ltQXJwTFNua2VOSzk1bmZTb2hNbVJQQVNVOERsSmZuS3p6SWlpSmFSLUQycEJGVkZ5anhCUlBFTzlqVlV5SUt4ZE0tWkxmVlByZl93VUhUNWtrTzd0N0pBMUhrWWNrRjlJUWxMQzZPUU9uMA?oc=5" target="_blank" rel="noopener">World Cup nations slam UEFA chief for ‘disappointing’ 48-team criticism - Al Jazeera rss <a href="https://news.google.com/rss/articles/CBMivgFBVV95cUxOM2VBT092eUFLYUV2YnRQYXZqR3dWLTNlNnJHQ3AzdWJkUl9lbVprdzRfRnl5QWRac3N1U3M2bG8xeGctQmhlaGxpajBFZi1pSGlWQXJ0c2pvaTg4b1NsVFQzaXB3NnlKWHF3NTBFRDZSTDdXNDFaOFQ5VjhVaHJ3c2xKQ1FsSVc3aXRfRnhlQjlaTFBGV3FXQzhKNjFyOHpRbFF3OGVrb1ZNOV9oLXVpQ0FSd0tydUNrX1BPbnlR0gHDAUFVX3lxTE1WS1JyRktTU2Q2WWZpV21yc3FuVU5NUzV5ZHBZS2NYZHV1ZzNBQ0RuTFNOdE04NGQ5SjhOcmQwUnZ4bElaeEFXLUVVMnlEWmZheEJ5MnVJdGNpUUt1NXVINUNyQURIYWFLVFVfSktsd3hBTDJJMVhQYXZ3aTI1aENLb2N4cEI5VnpCeG9lYVo3R0lJdW1FNXdhc0w3UU9sTVlSMG54eFB0N1dYWUxPampjM0Niak5rTzJXQVhqSHA2a3hhNA?oc=5" target="_blank" rel="noopener">Tom Cruise, David Beckham, Katy Perry and more celebrities spotted at 2026 FIFA World Cup matches - Fox News rss Iran team arrive in US for World Cup opener as the two nations reach peace deal - Reuters rss 
 
 Sources: gamma-api.polymarket.com · rsshub.app (Google-News fallback). Re-run a source cell to refresh. Rendered by the world-cup-2026 Spur App.